# Bootstrap Manifest
Run this once to create manifests from your existing files on Drive.
After this, the pipeline notebooks will keep manifests up to date automatically.

In [ ]:
import os, json, hashlib, datetime
from google.colab import drive

drive.mount('/content/drive')

SPARKYLLM = "/content/drive/MyDrive/sparkyllm"
BASE_PATH = os.path.join(SPARKYLLM, "datasets_pretrain")
MANIFEST_DIR = os.path.join(SPARKYLLM, "manifests")
os.makedirs(MANIFEST_DIR, exist_ok=True)

def file_hash(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while chunk := f.read(8192):
            h.update(chunk)
    return h.hexdigest()

def file_info(path):
    if not os.path.exists(path):
        return {"path": path, "exists": False}
    return {
        "path": path,
        "size_mb": round(os.path.getsize(path) / 1024 / 1024, 2),
        "sha256": file_hash(path),
    }

# ---- 1. Consolidation manifest (scan all data_* folders) ----
data_dirs = sorted([
    d for d in os.listdir(BASE_PATH)
    if d.startswith("data_") and os.path.isdir(os.path.join(BASE_PATH, d))
])

all_sources = {}
for dirname in data_dirs:
    dirpath = os.path.join(BASE_PATH, dirname)
    files = []
    for f in sorted(os.listdir(dirpath)):
        if f.endswith(".txt"):
            files.append({"filename": f, **file_info(os.path.join(dirpath, f))})
    all_sources[dirname] = files

all_hashes = "".join(
    f["sha256"] for files in all_sources.values() for f in files if "sha256" in f
)
combined_hash = hashlib.sha256(all_hashes.encode()).hexdigest()[:16]
total_files = sum(len(files) for files in all_sources.values())

consol = {
    "stage": "consolidation",
    "created": datetime.datetime.now().isoformat(),
    "sources_hash": combined_hash,
    "num_data_dirs": len(data_dirs),
    "num_source_files": total_files,
    "data_dirs": {dirname: files for dirname, files in all_sources.items()},
    "output": file_info(os.path.join(BASE_PATH, "training_data_long.txt")),
}
with open(os.path.join(MANIFEST_DIR, "consolidation_latest.json"), "w") as f:
    json.dump(consol, f, indent=2)
print(f"Consolidation: {len(data_dirs)} folders, {total_files} files, sources_hash={combined_hash}")

# ---- 2. Tokenization manifest ----
TOK_DIR = os.path.join(BASE_PATH, "tokenizer_out")
meta_path = os.path.join(TOK_DIR, "train_long_meta.json")
with open(meta_path) as f:
    meta = json.load(f)

tok_info = file_info(os.path.join(TOK_DIR, "tokenizer.json"))
tokenization_id = tok_info["sha256"][:16]

tok = {
    "stage": "tokenization",
    "created": datetime.datetime.now().isoformat(),
    "tokenization_id": tokenization_id,
    "parent_sources_hash": combined_hash,
    "tokenizer": tok_info,
    "token_bin": file_info(os.path.join(TOK_DIR, "train_long.bin")),
    "config": {
        "type": "byte-level-bpe",
        "vocab_size": meta["vocab_size"],
        "trained_new": True,
    },
    "stats": {
        "num_tokens": meta["num_tokens"],
        "num_documents": meta["num_documents"],
    },
}
with open(os.path.join(MANIFEST_DIR, "tokenization_latest.json"), "w") as f:
    json.dump(tok, f, indent=2)
print(f"Tokenization: {meta['num_tokens']:,} tokens, tokenization_id={tokenization_id}")

# ---- 3. Training manifest ----
CKPT = os.path.join(SPARKYLLM, "checkpoints", "gpt_medium_phase2.pth")
if os.path.exists(CKPT):
    train = {
        "stage": "training",
        "created": datetime.datetime.now().isoformat(),
        "checkpoint": file_info(CKPT),
        "tokenization_id": tokenization_id,
        "config": {
            "block_size": 768, "embed_dim": 1280, "num_layers": 24,
            "num_heads": 20, "ff_hidden_dim": 5120, "dropout": 0.1,
            "vocab_size_padded": 15040,
            "batch_size": 32, "grad_accum_steps": 8, "epochs": 5,
            "max_lr": 3e-4, "min_lr": 3e-5, "warmup_steps": 100,
            "weight_decay": 0.1, "precision": "bfloat16",
        },
        "notes": "First run with new tokenizer, trained from scratch",
    }
    with open(os.path.join(MANIFEST_DIR, "training_latest.json"), "w") as f:
        json.dump(train, f, indent=2)
    print(f"Training: {train['checkpoint'].get('size_mb', '?')} MB checkpoint")
else:
    print(f"No checkpoint at {CKPT}, skipping training manifest.")

print(f"\nAll manifests saved to: {MANIFEST_DIR}")